In [13]:
######################################################################################################################
# Notebook: 05_BM25_2Pasadas_LLM_Rewrite_Evaluacion
# Autor del código: Vladimir Molleapasa Gutierrez
# Fecha de generación: 17/01/2026
# Código generado con asistencia de ChatGPT 5.2 Thinking
# Prompt original: "Genera un notebook Jupyter para un experimento reproducible de recuperación arancelaria NANDINA 
#                   con BM25 + 2 pasadas + LLM (Ollama) rewrite. Debe leer toda la configuración desde:
#                   C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src\configs\experiment_config.json 
#                  (paths y parámetros bm25.* y llm.*, incluyendo paths.base_dir, dataset_path, bm25_index_path, 
#                   runs_dir, top_n, top_m, allowed_k, max_query_terms, k_list, y llm.base_url, llm.model, llm.temperature).
#                  El notebook debe importar BM25Index, normalize_text, tokenize_es, sha256_file desde bm25_index.py 
#                  ubicado en Código\src\ (o Código\scr\, agregando la ruta correcta al sys.path).
#                  Pipeline: (1) BM25 puro con descripción original a TOP_N; (2) 2 pasadas: primera pasada BM25 
#                  a TOP_M, construir allowed_terms desde los textos recuperados (top_m), llamar a Ollama /api/chat 
#                  con structured outputs (schema JSON) para reescribir query_bm25, 
#                  aplicar sanitización determinística (sin AND/OR/NOT, sin signos, 
#                  sin stopwords, sin tokens con dígitos/unidades, sin tokens bloqueados, sin OOV vs allowed_terms) y 
#                  luego aplicar una regla fija anti-deriva por anclas: extraer anchors_raw determinísticamente del RAW 
#                  y exigir que el query final contenga ≥ MIN_ANCHORS; si no, inyectar anclas al inicio; 
#                  si aún no, fallback a “anchors-only”.
#                  El notebook debe crear un directorio runs/bm25_2pass_llm_<timestamp>/ y 
#                  guardar tres artefactos: results.csv (métricas por fila + top-N), 
#                  llm_rewrites.jsonl (log por fila: RAW, top_m_first_pass_codes, allowed_terms_sample, 
#                  respuesta LLM raw, query_sanitizado, query_final, auditoría de
#                  anclas/removed tokens, validación/fallback), y run_metadata.json 
#                  (hashes SHA-256 de config/dataset/índice, plataforma/Python, parámetros, 
#                  y métricas agregadas: MRR y Acc@K para K en k_list).
#                  Manejo robusto: si Ollama no responde, error accionable; si JSON inválido, 
#                  fallback a RAW y log de motivo; evaluación determinística (temperatura 0).
#                  Documentación: primera celda en comentarios (no markdown) explicando propósito, 
#                  entradas/salidas, pasos de ejecución, trazabilidad; comentarios en español y estilo académico."
# Autor del prompt: Vladimir Molleapasa Gutierrez
# Ajustes y validación: Vladimir Molleapasa Gutierrez
# Uso académico, con revisión propia del autor.
# Licencia: Uso académico, no comercial.
######################################################################################################################

# Propósito
# ---------
# Ejecutar y evaluar un baseline de recuperación BM25 para clasificación arancelaria
# (NANDINA a 8 dígitos) con una estrategia de 2 pasadas:
#   (1) BM25 (consulta original) -> Top-M candidatos
#   (2) LLM local (Ollama) reescribe una consulta corta restringida por vocabulario
#       derivado del Top-M; la consulta se sanea (sanitization) y se ejecuta un
#       segundo BM25 -> Top-N final.
#
# Reproducibilidad
# ----------------
# - Lee parámetros desde un archivo JSON de configuración del experimento.
# - Genera un directorio de corrida (runs/<run_id>/) con:
#     - results.csv
#     - llm_rewrites.jsonl
#     - run_metadata.json
#     - copia del experiment_config.json
# - Registra hashes SHA-256 del índice, del dataset y del config (trazabilidad).
#
# Requisitos
# ----------
# - Tener un índice BM25 serializado (pickle) generado previamente (notebook 04).
# - Tener Ollama ejecutándose en local (por defecto http://127.0.0.1:11434).
# - Tener un dataset CSV/XLSX con columnas: descripcion, nandina

In [14]:
# =========================
# Imports y utilidades base
# =========================

import os
import sys
import json
import re
import time
import shutil
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np

try:
    import pandas as pd
except Exception as e:
    raise RuntimeError("Se requiere pandas para este notebook (instálalo en tu entorno).") from e

In [15]:
# =====================================
# Cargar configuración y preparar corrida
# =====================================
# Esta celda:
# - Carga el archivo experiment_config.json desde src\configs
# - Valida claves mínimas requeridas
# - Define parámetros globales del experimento (BM25 + LLM)
# - Crea el directorio de corrida (runs/...) y copia el config
# - Calcula SHA-256 de config/dataset/índice para trazabilidad
# - Define rutas de artefactos (results.csv, llm_rewrites.jsonl, run_metadata.json)

import os
import json
import shutil
import hashlib
import platform
import sys
from datetime import datetime

# Ruta fija (fuente de verdad)
CONFIG_PATH = r"C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src\configs\experiment_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"""No se encontró el archivo de configuración.
CONFIG_PATH={CONFIG_PATH}

Sugerencia:
- Verifica que el archivo exista en esa ruta, o
- Ajusta CONFIG_PATH en esta celda.
""")

# Utilidades de hash (lectura por chunks para archivos grandes)
def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

# Cargar config (y conservar texto para hash)
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CFG_TEXT = f.read()
CFG = json.loads(CFG_TEXT)

# Validación mínima del esquema
for k in ["paths", "bm25", "llm"]:
    if k not in CFG:
        raise KeyError(f"Falta la clave '{k}' en el config: {CONFIG_PATH}")

for k in ["bm25_index_path", "dataset_path", "runs_dir"]:
    if k not in CFG["paths"]:
        raise KeyError(f"Falta la clave paths.{k} en el config: {CONFIG_PATH}")

for k in ["top_n", "top_m", "allowed_k", "max_query_terms", "k_list"]:
    if k not in CFG["bm25"]:
        raise KeyError(f"Falta la clave bm25.{k} en el config: {CONFIG_PATH}")

for k in ["base_url", "model", "temperature"]:
    if k not in CFG["llm"]:
        raise KeyError(f"Falta la clave llm.{k} en el config: {CONFIG_PATH}")

# Rutas de insumo
BM25_INDEX_PATH = CFG["paths"]["bm25_index_path"]
DATASET_PATH = CFG["paths"]["dataset_path"]
RUNS_DIR = CFG["paths"]["runs_dir"]

# Parámetros BM25
TOP_N = int(CFG["bm25"]["top_n"])
TOP_M = int(CFG["bm25"]["top_m"])
ALLOWED_K = int(CFG["bm25"]["allowed_k"])
MAX_QUERY_TERMS = int(CFG["bm25"]["max_query_terms"])
K_LIST = list(CFG["bm25"]["k_list"])

# Parámetros LLM (Ollama)
OLLAMA_URL = CFG["llm"]["base_url"]
LLM_MODEL = CFG["llm"]["model"]
LLM_TEMPERATURE = float(CFG["llm"]["temperature"])

# Validaciones de coherencia (consistencia, no tuning)
if TOP_M < TOP_N:
    raise ValueError(f"Inconsistencia: TOP_M ({TOP_M}) debe ser >= TOP_N ({TOP_N}).")
if MAX_QUERY_TERMS <= 0:
    raise ValueError("MAX_QUERY_TERMS debe ser > 0.")
if not isinstance(K_LIST, list) or len(K_LIST) == 0:
    raise ValueError("K_LIST debe ser una lista no vacía (ej.: [1,3,5,10]).")

# Verificación de existencia de insumos
if not os.path.exists(BM25_INDEX_PATH):
    raise FileNotFoundError(f"No se encontró el índice BM25 en: {BM25_INDEX_PATH}")
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"No se encontró el dataset/devset en: {DATASET_PATH}")

# Crear directorio de corrida
os.makedirs(RUNS_DIR, exist_ok=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = os.path.join(RUNS_DIR, f"bm25_2pass_llm_{run_id}")
os.makedirs(OUT_DIR, exist_ok=True)

# Rutas de artefactos
OUT_RESULTS = os.path.join(OUT_DIR, "results.csv")
OUT_LLM_LOG = os.path.join(OUT_DIR, "llm_rewrites.jsonl")
OUT_META = os.path.join(OUT_DIR, "run_metadata.json")

# Copiar config al run (evidencia)
config_copy_path = os.path.join(OUT_DIR, "experiment_config.json")
shutil.copy2(CONFIG_PATH, config_copy_path)

# Hashes de trazabilidad
CFG_SHA256 = sha256_text(CFG_TEXT)
INDEX_SHA256 = sha256_file(BM25_INDEX_PATH)
DATASET_SHA256 = sha256_file(DATASET_PATH)

MIN_ANCHORS = 2          # regla fija de fidelidad
ANCHOR_TOP_K = 6         # cuántos términos candidatos a ancla se extraen del RAW
DROP_NUMERIC_TOKENS = True

STOPWORDS_ES = {
    "de","del","la","las","el","los","un","una","unos","unas",
    "y","o","u","para","por","con","sin","en","al","a",
    "que","como","se","su","sus","es","son","entre","incluye","incluso"
}


# Metadata base (se completa al final con métricas)
RUN_METADATA_BASE = {
    "run_id": run_id,
    "created_at_local": datetime.now().isoformat(),
    "platform": {
        "python": sys.version,
        "os": platform.platform(),
        "machine": platform.machine(),
    },
    "inputs": {
        "config_path": CONFIG_PATH,
        "config_copy_path": config_copy_path,
        "config_sha256": CFG_SHA256,
        "bm25_index_path": BM25_INDEX_PATH,
        "bm25_index_sha256": INDEX_SHA256,
        "dataset_path": DATASET_PATH,
        "dataset_sha256": DATASET_SHA256,
    },
    "params": {
        "bm25": {
            "top_n": TOP_N,
            "top_m": TOP_M,
            "allowed_k": ALLOWED_K,
            "max_query_terms": MAX_QUERY_TERMS,
            "k_list": K_LIST
        },
        "llm": {
            "base_url": OLLAMA_URL,
            "model": LLM_MODEL,
            "temperature": LLM_TEMPERATURE
        }
    },
    "artifacts": {
        "out_dir": OUT_DIR,
        "results_csv": OUT_RESULTS,
        "llm_log_jsonl": OUT_LLM_LOG,
        "run_metadata_json": OUT_META
    }
}

print("OK: config cargado y corrida preparada.")
print(" - CONFIG_PATH :", CONFIG_PATH)
print(" - OUT_DIR     :", OUT_DIR)
print(" - OUT_RESULTS :", OUT_RESULTS)
print(" - OUT_LLM_LOG :", OUT_LLM_LOG)
print(" - OUT_META    :", OUT_META)
print(" - CFG_SHA256  :", CFG_SHA256)
print(" - INDEX_SHA256:", INDEX_SHA256)
print(" - DATASET_SHA256:", DATASET_SHA256)


OK: config cargado y corrida preparada.
 - CONFIG_PATH : C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src\configs\experiment_config.json
 - OUT_DIR     : C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\runs\bm25_2pass_llm_20260117_185139
 - OUT_RESULTS : C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\runs\bm25_2pass_llm_20260117_185139\results.csv
 - OUT_LLM_LOG : C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\runs\bm25_2pass_llm_20260117_185139\llm_rewrites.jsonl
 - OUT_META    : C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\runs\bm25_2pass_llm_20260117_185139\run_metadata.json
 - CFG_SHA256  : afd7f11d6efe66776f1041b73a80b7fbcf62199413fc6ae8ec99a75b24119036
 - INDEX_SHA256: fd5eb111f95dc4de09f1a47fdb1117f455a5caeed96548a25219664a28857b6b
 - DATASET_SHA256: 5917b32588665883460531d240682d0a47a093b02c3da331ae2d52a3516533be


In [16]:
# =========================
# Preparar entorno (src/)
# =========================
# Esta celda:
# - Verifica que la configuración (CFG) ya fue cargada
# - Obtiene BASE_DIR desde CFG["paths"]["base_dir"]
# - Agrega la carpeta src/ al sys.path para permitir imports locales (bm25_index.py)
# - Importa BM25Index y utilidades desde bm25_index.py

import os
import sys

# Verificación: la configuración debe haberse cargado en una celda previa
if "CFG" not in globals():
    raise RuntimeError(
        "No se encontró la variable global 'CFG'. "
        "Ejecuta primero la celda: 'Cargar configuración y preparar corrida'."
    )

# BASE_DIR proviene del archivo experiment_config.json (paths.base_dir)
BASE_DIR = CFG["paths"].get("base_dir", "").strip()
if not BASE_DIR:
    raise KeyError("Falta 'paths.base_dir' en el archivo de configuración (CFG).")

# Carpeta de código fuente (src/)
SCR_DIR = os.path.join(BASE_DIR, "src")

# Compatibilidad: si tu proyecto usa 'scr/' en lugar de 'src/', se usa como alternativa
if not os.path.isdir(SCR_DIR):
    alt_dir = os.path.join(BASE_DIR, "scr")
    if os.path.isdir(alt_dir):
        SCR_DIR = alt_dir
    else:
        raise FileNotFoundError(
            f"No se encontró la carpeta de código fuente.\n"
            f"Probadas:\n"
            f"- {os.path.join(BASE_DIR, 'src')}\n"
            f"- {os.path.join(BASE_DIR, 'scr')}\n"
            f"Verifica 'paths.base_dir' en tu experiment_config.json y tu estructura de carpetas."
        )

# Inyección controlada al path para imports locales
if SCR_DIR not in sys.path:
    sys.path.insert(0, SCR_DIR)

from bm25_index import BM25Index, normalize_text, tokenize_es, sha256_file

print("OK: bm25_index importado desde:", SCR_DIR)


OK: bm25_index importado desde: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\src


In [17]:
# ==============================
# Cargar dataset de evaluación
# ==============================

def is_nandina8(x: Any) -> bool:
    return bool(re.fullmatch(r"\d{8}", str(x).strip()))

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"""No se encontró el dataset.
DATASET_PATH={DATASET_PATH}

Sugerencia: ajusta dataset_path en experiment_config.json.
""")

ext = os.path.splitext(DATASET_PATH)[1].lower()
if ext == ".csv":
    df = pd.read_csv(DATASET_PATH)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(DATASET_PATH)
else:
    raise ValueError(f"Extensión no soportada: {ext}. Usa CSV o Excel.")

# Normalización de nombres de columnas
cols = {c.lower().strip(): c for c in df.columns}
if "descripcion" not in cols or "nandina" not in cols:
    raise KeyError(f"El dataset debe contener columnas 'descripcion' y 'nandina'. Columnas encontradas: {list(df.columns)}")

df = df.rename(columns={cols["descripcion"]: "descripcion", cols["nandina"]: "nandina"})

# Filtrado: NANDINA a 8 dígitos
before = len(df)
df = df[df["nandina"].apply(is_nandina8)].copy()
df["nandina"] = df["nandina"].astype(str)
df = df.reset_index(drop=True)

after = len(df)
print("OK: dataset cargado.")
print(" - ruta:", DATASET_PATH)
print(" - filas (antes):", before)
print(" - filas (válidas 8 dígitos):", after)

OK: dataset cargado.
 - ruta: C:\Users\VLADIMIR\OneDrive\Documentos\Tesis UPEU\Código\data\processed\devset_validacion_intermedia.csv
 - filas (antes): 13
 - filas (válidas 8 dígitos): 11


In [18]:
# ==============================
# Cargar índice BM25 serializado
# ==============================

import pickle

if not os.path.exists(BM25_INDEX_PATH):
    raise FileNotFoundError(f"""No se encontró el índice BM25 (pickle).
BM25_INDEX_PATH={BM25_INDEX_PATH}

Sugerencia: ejecuta primero el notebook de indexación (04_...) y verifica la ruta.
""")

with open(BM25_INDEX_PATH, "rb") as f:
    bm25 = pickle.load(f)

if not isinstance(bm25, BM25Index):
    raise TypeError(
        "El objeto cargado no es una instancia de BM25Index. "
        "Verifica que el índice haya sido generado con el módulo bm25_index.py"
    )

print("OK: índice BM25 cargado.")
print(" - docs:", len(bm25.doc_ids))
print(" - vocab:", len(bm25.idf))

OK: índice BM25 cargado.
 - docs: 7644
 - vocab: 5646


In [19]:
# ==========================
# Utilidades de recuperación
# ==========================

def bm25_retrieve(query: str, top_n: int) -> List[Dict[str, Any]]:
    """Ejecuta BM25 y retorna hits con código y texto."""
    scored = bm25.score(query, top_n=top_n)
    hits: List[Dict[str, Any]] = []
    for doc_idx, score in scored:
        code = bm25.doc_ids[doc_idx]
        text = bm25.doc_texts[doc_idx]
        hits.append({
            "doc_idx": int(doc_idx),
            "code": str(code),
            "score": float(score),
            "text": str(text),
        })
    return hits

def build_allowed_terms_from_hits(hits: List[Dict[str, Any]], top_k: int, extra_text: str = "") -> List[str]:
    """Construye vocabulario permitido a partir de los textos del Top-M."""
    from collections import Counter

    cnt = Counter()
    if extra_text:
        cnt.update(tokenize_es(extra_text))

    for h in hits:
        cnt.update(tokenize_es(h.get("text", "")))

    terms = [t for t, _ in cnt.most_common(top_k)]
    return terms

# Términos bloqueados (ruido técnico que suele sesgar hacia subpartidas incorrectas)
# Nota: si deseas formalizar esto, mueve este listado al experiment_config.json.
BLOCKED_TERMS = {
    # hardware/marketing
    "intel", "amd", "ryzen", "core", "i3", "i5", "i7", "i9",
    "ram", "ssd", "hdd", "usb", "hdmi", "wifi", "bluetooth",
    "led", "lcd", "oled",
    # unidades y formatos
    "gb", "tb", "mhz", "ghz", "hz", "pulgadas", "pulgada", "cm", "mm",
    # tokens genéricos poco discriminantes
    "portatil", "externo", "interno"
}

BOOL_OPS = {"and", "or", "not"}
MIN_QUERY_TERMS = 2

def sanitize_llm_query(
    q: str,
    allowed_terms_set: set,
    blocked_terms_set: set,
    max_terms: int
) -> Tuple[str, Dict[str, Any]]:
    """Sanea una consulta del LLM para que sea ejecutable por BM25 con restricciones.

    Reglas determinísticas:
    - Elimina operadores booleanos (AND/OR/NOT) y símbolos.
    - Elimina términos bloqueados.
    - Elimina tokens fuera del vocabulario permitido (OOV) derivado del Top-M.
    - Deduplica conservando orden y recorta a max_terms.
    """
    q0 = (q or "").strip()

    # Limpiar símbolos comunes
    q0 = re.sub(r"[(){}\[\],;:|/\"']", " ", q0)
    q0 = re.sub(r"(AND|OR|NOT)", " ", q0, flags=re.IGNORECASE)
    q0 = re.sub(r"\s+", " ", q0).strip()

    toks = tokenize_es(q0)

    removed = {"bool_ops": [], "blocked": [], "oov": []}
    kept: List[str] = []

    for t in toks:
        if t in BOOL_OPS:
            removed["bool_ops"].append(t)
            continue
        if t in blocked_terms_set:
            removed["blocked"].append(t)
            continue
        if t not in allowed_terms_set:
            removed["oov"].append(t)
            continue
        kept.append(t)

    # deduplicación estable
    seen = set()
    kept = [t for t in kept if not (t in seen or seen.add(t))]
    kept = kept[:max_terms]

    q_san = " ".join(kept).strip()

    audit = {
        "kept_terms": kept,
        "removed": removed,
        "min_terms_ok": len(kept) >= MIN_QUERY_TERMS,
    }
    return q_san, audit

import re
from typing import List, Set, Dict, Any, Tuple


def is_numeric_like(tok: str) -> bool:
    # dígitos, decimales, unidades tipo 3.2, 0.94, 512, 14
    return bool(re.search(r"\d", tok))


def extract_anchors_from_raw(raw: str, allowed_terms_set: Set[str], top_k: int = ANCHOR_TOP_K) -> List[str]:
    """
    Extrae anclas determinísticas desde la descripción original:
    - normaliza y tokeniza
    - elimina stopwords
    - opcionalmente elimina tokens con dígitos
    - filtra por vocabulario permitido (allowed_terms)
    - retorna los primeros top_k tokens (orden estable)
    """
    toks = tokenize_es(normalize_text(raw))
    anchors = []
    for t in toks:
        if t in STOPWORDS_ES:
            continue
        if DROP_NUMERIC_TOKENS and is_numeric_like(t):
            continue
        if t not in allowed_terms_set:
            continue
        anchors.append(t)

    # deduplicar conservando orden
    seen = set()
    anchors = [t for t in anchors if not (t in seen or seen.add(t))]

    return anchors[:top_k]

def enforce_anchor_policy(
    raw: str,
    query_sanitizado: str,
    anchors: List[str],
    max_terms: int = 12,
    min_anchors: int = MIN_ANCHORS
) -> Tuple[str, Dict[str, Any]]:
    """
    Regla fija: el query final debe contener al menos min_anchors anclas del RAW.
    Si no las contiene, se inyectan; si aun así queda corto, se hace fallback a anchors-only.
    """
    q_toks = tokenize_es(normalize_text(query_sanitizado))
    q_set = set(q_toks)

    anchors_in = [a for a in anchors if a in q_set]
    anchors_missing = [a for a in anchors if a not in q_set]

    # Si cumple, no se toca
    if len(anchors_in) >= min_anchors:
        return query_sanitizado, {
            "anchor_ok": True,
            "anchors_raw": anchors,
            "anchors_in_query": anchors_in,
            "anchors_missing": anchors_missing,
            "action": "keep"
        }

    # Inyectar anclas faltantes al inicio (para subir peso en BM25)
    injected = anchors_missing + q_toks
    # dedup estable
    seen = set()
    injected = [t for t in injected if not (t in seen or seen.add(t))]
    injected = injected[:max_terms]
    q_injected = " ".join(injected).strip()

    q_inj_toks = tokenize_es(normalize_text(q_injected))
    anchors_in2 = [a for a in anchors if a in set(q_inj_toks)]

    if len(anchors_in2) >= min_anchors:
        return q_injected, {
            "anchor_ok": True,
            "anchors_raw": anchors,
            "anchors_in_query": anchors_in2,
            "anchors_missing": [a for a in anchors if a not in set(q_inj_toks)],
            "action": "inject"
        }

    # Fallback determinístico: solo anclas
    q_fallback = " ".join(anchors[:max_terms]).strip()
    return q_fallback, {
        "anchor_ok": False,
        "anchors_raw": anchors,
        "anchors_in_query": anchors_in2,
        "anchors_missing": [a for a in anchors if a not in set(q_inj_toks)],
        "action": "fallback_anchors_only"
    }


In [20]:
# ==============================
# Cliente Ollama + prompt de reescritura
# ==============================

import json
import re
import urllib.request
from typing import List, Dict, Any, Set, Tuple

# ------------------------------
# Parámetros fijos de fidelidad
# ------------------------------
MIN_QUERY_TERMS = 2       # mínimo de términos para ejecutar BM25
MIN_ANCHORS = 2           # mínimo de anclas del RAW que deben estar presentes en el query final
ANCHOR_TOP_K = 6          # cuántas anclas candidatas se extraen del RAW
DROP_NUMERIC_TOKENS = True

# Lista fija y acotada de stopwords para evitar ruido en el query
STOPWORDS_ES = {
    "de","del","la","las","el","los","un","una","unos","unas",
    "y","o","u","para","por","con","sin","en","al","a",
    "que","como","se","su","sus","es","son","entre","incluso",
    "incluida","incluido","incluyen","incluye"
}

def is_numeric_like(tok: str) -> bool:
    """Detecta tokens con dígitos (incluye decimales y unidades)."""
    return bool(re.search(r"\d", tok))

def extract_anchors_from_raw(raw: str, allowed_terms_set: Set[str], top_k: int = ANCHOR_TOP_K) -> List[str]:
    """
    Extrae anclas determinísticas desde la descripción original:
    - normaliza y tokeniza
    - elimina stopwords
    - opcionalmente elimina tokens con dígitos
    - filtra por vocabulario permitido (allowed_terms)
    - retorna los primeros top_k tokens (orden estable)
    """
    toks = tokenize_es(normalize_text(raw))
    anchors = []
    for t in toks:
        if t in STOPWORDS_ES:
            continue
        if DROP_NUMERIC_TOKENS and is_numeric_like(t):
            continue
        if t not in allowed_terms_set:
            continue
        anchors.append(t)

    # deduplicación estable
    seen = set()
    anchors = [t for t in anchors if not (t in seen or seen.add(t))]

    return anchors[:top_k]

def enforce_anchor_policy(
    raw: str,
    query_sanitizado: str,
    anchors: List[str],
    max_terms: int,
    min_anchors: int = MIN_ANCHORS
) -> Tuple[str, Dict[str, Any]]:
    """
    Regla fija anti-deriva:
    - El query final debe contener al menos min_anchors anclas extraídas del RAW.
    - Si no se cumple, se inyectan anclas faltantes al inicio del query.
    - Si aun no se cumple o el query queda insuficiente, fallback determinístico a 'anchors-only'.

    Retorna:
    - query_final (string)
    - auditoría (dict) con acción aplicada y cobertura de anclas.
    """
    q_toks = tokenize_es(normalize_text(query_sanitizado))
    q_set = set(q_toks)

    anchors_in = [a for a in anchors if a in q_set]
    anchors_missing = [a for a in anchors if a not in q_set]

    # Caso 1: cumple
    if len(anchors_in) >= min_anchors:
        return query_sanitizado, {
            "anchor_ok": True,
            "anchors_raw": anchors,
            "anchors_in_query": anchors_in,
            "anchors_missing": anchors_missing,
            "action": "keep"
        }

    # Caso 2: inyectar anclas faltantes al inicio (mejora peso BM25)
    injected = anchors_missing + q_toks
    seen = set()
    injected = [t for t in injected if not (t in seen or seen.add(t))]
    injected = injected[:max_terms]
    q_injected = " ".join(injected).strip()

    q_inj_set = set(tokenize_es(normalize_text(q_injected)))
    anchors_in2 = [a for a in anchors if a in q_inj_set]
    anchors_missing2 = [a for a in anchors if a not in q_inj_set]

    if len(anchors_in2) >= min_anchors:
        return q_injected, {
            "anchor_ok": True,
            "anchors_raw": anchors,
            "anchors_in_query": anchors_in2,
            "anchors_missing": anchors_missing2,
            "action": "inject"
        }

    # Caso 3: fallback determinístico a anclas
    q_fallback = " ".join(anchors[:max_terms]).strip()
    return q_fallback, {
        "anchor_ok": False,
        "anchors_raw": anchors,
        "anchors_in_query": anchors_in2,
        "anchors_missing": anchors_missing2,
        "action": "fallback_anchors_only"
    }

# ------------------------------
# Esquema de salida (structured)
# ------------------------------
OLLAMA_REWRITE_SCHEMA = {
  "type": "object",
  "required": [
    "producto_canonico",
    "atributos_discriminantes",
    "terminos_incluir",
    "terminos_excluir",
    "query_bm25",
    "confidence",
    "justificacion"
  ],
  "properties": {
    "producto_canonico": {"type": "string"},
    "atributos_discriminantes": {"type": "array", "items": {"type": "string"}},
    "terminos_incluir": {"type": "array", "items": {"type": "string"}},
    "terminos_excluir": {"type": "array", "items": {"type": "string"}},
    "query_bm25": {"type": "string"},
    "confidence": {"type": "number"},
    "justificacion": {"type": "string"}
  },
  "additionalProperties": False
}

def ollama_chat(prompt: str, model: str, temperature: float = 0.0) -> str:
    """
    Invoca Ollama (API /api/chat) y retorna el contenido del mensaje.

    Mejora operativa:
    - Timeout configurable (por defecto 600s)
    - Reintentos ante TimeoutError (por defecto 1 reintento)
    """
    timeout_s = int(CFG.get("llm", {}).get("timeout_seconds", 600))
    max_retries = int(CFG.get("llm", {}).get("max_retries", 1))

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "options": {"temperature": temperature},
        "format": OLLAMA_REWRITE_SCHEMA,
        "stream": False,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )

    last_err = None
    for attempt in range(max_retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout_s) as resp:
                out = json.loads(resp.read().decode("utf-8"))
            content = (out.get("message") or {}).get("content", "")
            if isinstance(content, str):
                return content
            return json.dumps(content, ensure_ascii=False)

        except TimeoutError as e:
            last_err = e
            # reintento controlado
            continue
        except Exception as e:
            # otros errores: no insistir (diagnóstico)
            raise ConnectionError(
                f"""No se pudo completar la llamada a Ollama.

OLLAMA_URL={OLLAMA_URL}
MODEL={model}
timeout_seconds={timeout_s}
max_retries={max_retries}

Verifica:
- Que Ollama esté ejecutándose y responda en /api/chat.
- Que el modelo exista localmente (ollama list) y esté descargado (ollama pull).
- Que no haya saturación de CPU/RAM/GPU.
- Que llm.base_url en experiment_config.json apunte a /api/chat.

Error original: {repr(e)}
"""
            ) from e

    # si se agotaron reintentos por timeout
    raise ConnectionError(
        f"""Ollama no respondió dentro del timeout.

OLLAMA_URL={OLLAMA_URL}
MODEL={model}
timeout_seconds={timeout_s}
max_retries={max_retries}

Sugerencias:
- Ejecutar un warm-up:  ollama run {model} "Hola"
- Aumentar timeout_seconds en experiment_config.json.
- Usar un modelo más pequeño si el hardware es limitado.

Error original: {repr(last_err)}
"""
    ) from last_err


def build_rewrite_prompt(descripcion_raw: str, allowed_terms: List[str], max_terms: int) -> str:
    """
    Construye un prompt de reescritura para consultas BM25.

    Principios:
    - Minimizar deriva semántica (no inventar términos).
    - Restringir formato: términos separados por espacios, sin operadores.
    - Forzar presencia de términos ancla del producto.
    """
    allowed_preview = ", ".join(allowed_terms[:200])

    prompt = f"""
Eres un asistente especializado en clasificación arancelaria.

Tarea:
- Reescribe la descripción de mercancía como un query corto para BM25.
- El query debe contener SOLO palabras separadas por espacios (sin AND/OR/NOT, sin paréntesis).
- Máximo {max_terms} términos.

Restricciones obligatorias:
- No inventes términos: usa solo palabras de la descripción y/o del vocabulario sugerido.
- Preserva términos ancla del producto (por ejemplo: "diodos", "led", "smd" si aparecen en la descripción).
- Evita números, medidas y versiones (GB, TB, pulgadas, 3.2, i5) salvo que sean esenciales.

Descripción original:
{descripcion_raw}

Vocabulario sugerido (muestra):
{allowed_preview}

Devuelve SOLO un JSON válido con el esquema solicitado.
""".strip()
    return prompt

def llm_rewrite_query(descripcion_raw: str, allowed_terms: List[str], max_terms: int) -> Dict[str, Any]:
    """
    Reescritura robusta para BM25:
    1) LLM propone query (JSON estructurado).
    2) Saneamiento determinístico: elimina operadores/símbolos, tokens bloqueados y OOV.
    3) Regla fija anti-deriva: el query final debe conservar anclas del RAW.
    4) Validación mínima y fallback determinístico si queda insuficiente.
    """
    prompt = build_rewrite_prompt(descripcion_raw, allowed_terms, max_terms)

    # 1) Llamada al LLM
    raw_llm = ollama_chat(prompt, model=LLM_MODEL, temperature=LLM_TEMPERATURE)

    # 2) Parse JSON (si falla, fallback)
    try:
        obj = json.loads(raw_llm)
    except Exception:
        obj = {
            "producto_canonico": "",
            "atributos_discriminantes": [],
            "terminos_incluir": [],
            "terminos_excluir": [],
            "query_bm25": descripcion_raw,
            "confidence": 0.0,
            "justificacion": "LLM no devolvió JSON válido; fallback a descripción original.",
        }

    # 3) Auditoría del LLM
    obj["_llm_raw"] = raw_llm
    obj["_query_llm_original"] = str(obj.get("query_bm25", "")).strip()

    # 4) Saneamiento determinístico (formato + control de vocabulario)
    allowed_set = set(allowed_terms)
    q_san, audit = sanitize_llm_query(
        obj["_query_llm_original"],
        allowed_terms_set=allowed_set,
        blocked_terms_set=BLOCKED_TERMS,
        max_terms=max_terms,
    )
    obj["_query_llm_sanitizado"] = q_san
    obj["_sanitize_audit"] = audit

    # 5) Regla fija anti-deriva: forzar anclas del RAW
    anchors = extract_anchors_from_raw(descripcion_raw, allowed_set, top_k=ANCHOR_TOP_K)
    q_final, anchor_audit = enforce_anchor_policy(
        raw=descripcion_raw,
        query_sanitizado=q_san,
        anchors=anchors,
        max_terms=max_terms,
        min_anchors=MIN_ANCHORS
    )
    obj["_anchors_raw"] = anchors
    obj["_anchor_audit"] = anchor_audit
    obj["_query_llm_final"] = q_final

    # 6) Consulta aplicada
    obj["query_bm25"] = q_final

    # 7) Validación mínima + fallback final
    final_toks = tokenize_es(normalize_text(obj["query_bm25"]))
    ok = len(final_toks) >= MIN_QUERY_TERMS
    reason = "ok" if ok else "too_few_terms_final"

    obj["_validation"] = {"ok": ok, "reason": reason}

    if not ok:
        obj["_validation"]["fallback_applied"] = True
        obj["query_bm25"] = descripcion_raw
    else:
        obj["_validation"]["fallback_applied"] = False

    obj["_llm"] = {"model": LLM_MODEL, "temperature": LLM_TEMPERATURE}
    return obj


In [22]:
# =====================================
# Evaluación: BM25 puro vs. 2 pasadas (LLM rewrite)
# =====================================

def rank_of_true(hits: List[Dict[str, Any]], true_code: str) -> int:
    """Retorna el rank (1..N) de true_code en hits; 0 si no aparece."""
    for i, h in enumerate(hits, start=1):
        if str(h.get("code")) == str(true_code):
            return i
    return 0

def mrr_from_rank(r: int) -> float:
    return 0.0 if r <= 0 else 1.0 / float(r)

def acc_at_k(r: int, k: int) -> float:
    return 1.0 if (r > 0 and r <= k) else 0.0

results_rows: List[Dict[str, Any]] = []

llm_log_path = os.path.join(OUT_DIR, "llm_rewrites.jsonl")
results_path = os.path.join(OUT_DIR, "results.csv")
meta_path = os.path.join(OUT_DIR, "run_metadata.json")

# Hashes para trazabilidad
config_sha256 = sha256_file(config_copy_path)
index_sha256 = sha256_file(BM25_INDEX_PATH)
dataset_sha256 = sha256_file(DATASET_PATH)

# Métricas agregadas
bm25_pure_mrr = []
bm25_rw_mrr = []
acc_pure = {k: [] for k in K_LIST}
acc_rw = {k: [] for k in K_LIST}

with open(llm_log_path, "w", encoding="utf-8") as flog:

    for row_idx, row in df.iterrows():
        descripcion = str(row["descripcion"])
        true_code = str(row["nandina"]).strip()

        # Baseline BM25 puro
        hits_pure = bm25_retrieve(descripcion, top_n=TOP_N)
        r_pure = rank_of_true(hits_pure, true_code)

        bm25_pure_mrr.append(mrr_from_rank(r_pure))
        for k in K_LIST:
            acc_pure[k].append(acc_at_k(r_pure, k))

        # 1ra pasada para vocabulario
        hits_m = bm25_retrieve(descripcion, top_n=TOP_M)
        allowed_terms = build_allowed_terms_from_hits(hits_m, top_k=ALLOWED_K, extra_text=descripcion)

        # Reescritura LLM + saneamiento
        rewrite_obj = llm_rewrite_query(descripcion, allowed_terms, max_terms=MAX_QUERY_TERMS)
        query_aplicado = rewrite_obj.get("query_bm25", descripcion)

        # 2da pasada
        hits_rw = bm25_retrieve(query_aplicado, top_n=TOP_N)
        r_rw = rank_of_true(hits_rw, true_code)

        bm25_rw_mrr.append(mrr_from_rank(r_rw))
        for k in K_LIST:
            acc_rw[k].append(acc_at_k(r_rw, k))

        # Log JSONL por fila (auditoría)
        log_obj = {
            "row": int(row_idx),
            "true_nandina": true_code,
            "descripcion_raw": descripcion,
            "top_m_first_pass_codes": [h["code"] for h in hits_m[:min(50, len(hits_m))]],
            "allowed_terms_sample": allowed_terms[:50],
            "rewrite": rewrite_obj,
        }
        flog.write(json.dumps(log_obj, ensure_ascii=False) + "\n")

        # Guardar fila para results.csv
        results_rows.append({
            "row": int(row_idx),
            "true_nandina": true_code,
            "descripcion": descripcion,
            "bm25_pure_rank": int(r_pure),
            "bm25_rw_rank": int(r_rw),
            "bm25_pure_top_codes": json.dumps([h["code"] for h in hits_pure], ensure_ascii=False),
            "bm25_rw_top_codes": json.dumps([h["code"] for h in hits_rw], ensure_ascii=False),
            "llm_query_original": rewrite_obj.get("_query_llm_original"),
            "llm_query_sanitizado": rewrite_obj.get("_query_llm_sanitizado"),
            "query_aplicado": query_aplicado,
            "rewrite_ok": bool(rewrite_obj.get("_validation", {}).get("ok", False)),
            "rewrite_reason": str(rewrite_obj.get("_validation", {}).get("reason")),
            "fallback": bool(rewrite_obj.get("_validation", {}).get("fallback_applied", False)),
        })

# Exportar resultados
res_df = pd.DataFrame(results_rows)
res_df.to_csv(results_path, index=False, encoding="utf-8")

summary = {
    "rows": int(len(df)),
    "bm25_pure_mrr_mean": float(np.mean(bm25_pure_mrr)) if bm25_pure_mrr else 0.0,
    "bm25_rw_mrr_mean": float(np.mean(bm25_rw_mrr)) if bm25_rw_mrr else 0.0,
}

for k in K_LIST:
    summary[f"bm25_pure_acc@{k}"] = float(np.mean(acc_pure[k])) if acc_pure[k] else 0.0
    summary[f"bm25_rw_acc@{k}"] = float(np.mean(acc_rw[k])) if acc_rw[k] else 0.0

metadata = {
    "run_id": run_id,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "paths": {
        "config": config_copy_path,
        "bm25_index": BM25_INDEX_PATH,
        "dataset": DATASET_PATH,
        "out_dir": OUT_DIR,
        "results": results_path,
        "llm_log": llm_log_path,
    },
    "sha256": {
        "config": config_sha256,
        "bm25_index": index_sha256,
        "dataset": dataset_sha256,
    },
    "params": {
        "TOP_N": TOP_N,
        "TOP_M": TOP_M,
        "ALLOWED_K": ALLOWED_K,
        "MAX_QUERY_TERMS": MAX_QUERY_TERMS,
        "K_LIST": K_LIST,
        "BLOCKED_TERMS": sorted(list(BLOCKED_TERMS)),
        "MIN_QUERY_TERMS": MIN_QUERY_TERMS,
    },
    "llm": {
        "url": OLLAMA_URL,
        "model": LLM_MODEL,
        "temperature": LLM_TEMPERATURE,
        "schema": "OLLAMA_REWRITE_SCHEMA_v1",
    },
    "summary": summary,
    "python": {
        "version": sys.version,
    }
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("OK: artefactos guardados.")
print(" - results:", results_path)
print(" - llm log:", llm_log_path)
print(" - meta:", meta_path)

print("\nRESUMEN:", json.dumps(summary, ensure_ascii=False, indent=2))

ConnectionError: Ollama no respondió dentro del timeout.

OLLAMA_URL=http://127.0.0.1:11434/api/chat
MODEL=llama3.1:8b
timeout_seconds=600
max_retries=1

Sugerencias:
- Ejecutar un warm-up:  ollama run llama3.1:8b "Hola"
- Aumentar timeout_seconds en experiment_config.json.
- Usar un modelo más pequeño si el hardware es limitado.

Error original: TimeoutError('timed out')


In [38]:
# ==============================
# Visualización rápida de rewrites
# ==============================

# Muestra por pantalla: RAW vs LLM_ORIG vs SANITIZADO vs APLICADO
# (Lectura desde el log JSONL generado en esta corrida)

with open(llm_log_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        rw = obj["rewrite"]
        val = rw.get("_validation", {})

        print("=" * 110)
        print(f"row={obj['row']} | true={obj['true_nandina']} | ok={val.get('ok')} | reason={val.get('reason')} | fallback={val.get('fallback_applied')}")
        print("RAW       :", obj["descripcion_raw"])
        print("LLM_ORIG  :", rw.get("_query_llm_original"))
        print("SANITIZADO:", rw.get("_query_llm_sanitizado"))
        print("APLICADO  :", rw.get("query_bm25"))

        if i >= 25:
            print("(Corte: mostrando solo las primeras 26 filas)")
            break

row=0 | true=84713000 | ok=True | reason=ok | fallback=False
RAW       : Máquina automática para tratamiento o procesamiento de datos, portátil, constituida al menos por unidad central, teclado y visualizador integrados; peso menor a 10 kg.
LLM_ORIG  : automatica tratamiento procesamiento datos portatil unidad_central teclado visualizador_integrados peso_menor_a_10_kg
SANITIZADO: automatica tratamiento procesamiento datos unidad central teclado visualizador integrados peso menor 10
APLICADO  : automatica tratamiento procesamiento datos unidad central teclado visualizador integrados peso menor 10
row=1 | true=10063000 | ok=True | reason=ok | fallback=False
RAW       : Arroz semiblanqueado o blanqueado, incluso pulido o glaseado, en sacos para venta mayorista.
LLM_ORIG  : arroz semiblanqueado blanqueado pulido glaseado sacos venta mayorista
SANITIZADO: arroz semiblanqueado blanqueado pulido glaseado sacos venta mayorista
APLICADO  : arroz semiblanqueado blanqueado pulido glaseado sacos v